# 77 — Gaussian Process with Tanimoto Kernel

GP regression is often the best method for small molecular datasets (< 10k compounds).
The Tanimoto kernel is the natural similarity measure for binary fingerprints.
GPs also output calibrated uncertainty — useful for ensemble weighting.

Training a full GP on 4k compounds with 2048-dim fingerprints is expensive O(n³),
so we use a sparse approximation (Nyström / inducing points) and sub-sampling.


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Kernel, ConstantKernel as C
from sklearn.preprocessing import StandardScaler

class TanimotoKernel(Kernel):
    """Tanimoto (Jaccard) kernel for binary fingerprints."""
    def __call__(self, X, Y=None, eval_gradient=False):
        X = np.atleast_2d(X).astype(np.float32)
        Y = X if Y is None else np.atleast_2d(Y).astype(np.float32)
        dot = X @ Y.T
        sx  = X.sum(1)[:,None]; sy = Y.sum(1)[None,:]
        K   = dot / np.maximum(sx + sy - dot, 1e-8)
        if eval_gradient:
            return K.astype(np.float64), np.empty((X.shape[0],X.shape[0],0))
        return K.astype(np.float64)
    def diag(self, X):
        return np.ones(X.shape[0])
    def is_stationary(self): return False
    def get_params(self, deep=True): return {}
    def set_params(self, **p): return self

print("Tanimoto kernel defined.")


Tanimoto kernel defined.


In [5]:
# Sparse GP via Nyström approximation on inducing points
# Select 500 inducing points (diverse by MaxMin selection)
N_INDUCING = 500

def maxmin_select(fps, n):
    """MaxMin diversity selection of n inducing points."""
    selected = [0]
    min_dists = np.full(len(fps), np.inf)
    for _ in range(n-1):
        last = fps[selected[-1]]
        dot_l = fps @ last
        sl = last.sum(); sr = fps.sum(1)
        sim = dot_l / np.maximum(sl + sr - dot_l, 1e-8)
        dist = 1 - sim
        min_dists = np.minimum(min_dists, dist)
        min_dists[selected] = -1
        selected.append(int(np.argmax(min_dists)))
    return np.array(selected)

print(f"Selecting {N_INDUCING} inducing points by MaxMin diversity...")
inducing_idx = maxmin_select(fps_tr, N_INDUCING)
X_ind = fps_tr[inducing_idx].astype(np.float64)
print(f"Inducing points selected: {X_ind.shape}")


Selecting 500 inducing points by MaxMin diversity...


Inducing points selected: (500, 2048)


In [6]:
# Nystrom GP: fit GP on inducing points, predict via Nystrom approximation
kernel = C(1.0) * TanimotoKernel()
gp = GaussianProcessRegressor(kernel=kernel, alpha=0.5, n_restarts_optimizer=0,
                               normalize_y=True)

print("Fitting GP on inducing points...", flush=True)
gp.fit(X_ind, y_tr[inducing_idx])
print(f"Kernel params: {gp.kernel_}")

# Scaffold CV with Nyström GP
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_gp = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # Re-select inducing points from fold-train only
    n_ind_fold = min(N_INDUCING, len(tr_idx))
    ind_fold_local = maxmin_select(fps_tr[tr_idx], n_ind_fold)
    ind_fold_global = tr_idx[ind_fold_local]
    X_ind_f = fps_tr[ind_fold_global].astype(np.float64)
    y_ind_f = y_tr[ind_fold_global]

    gp_f = GaussianProcessRegressor(kernel=C(1.0)*TanimotoKernel(),
                                     alpha=0.5, normalize_y=True)
    gp_f.fit(X_ind_f, y_ind_f)
    oof_gp[va_idx] = gp_f.predict(fps_tr[va_idx].astype(np.float64))
    print(f"  fold {fold+1} RAE={rae(y_tr[va_idx], oof_gp[va_idx]):.4f}", flush=True)

m_gp = full_metrics(y_tr, oof_gp, cliff_pairs, "GP_tanimoto")
m_gp_a = full_metrics(y_tr[active_mask], oof_gp[active_mask], "GP_tanimoto [active]")
print("\n" + pd.DataFrame([m_gp, m_gp_a], index=["overall","active"]).round(4).to_string())
oof = oof_gp


Fitting GP on inducing points...


Kernel params: 0.717**2 * TanimotoKernel()

=== Scaffold 5-fold CV ===


  fold 1 RAE=0.7328


  fold 2 RAE=0.7885


  fold 3 RAE=0.7855


  fold 4 RAE=0.7985


  fold 5 RAE=0.8123


  [GP_tanimoto] RAE=0.7781 MAE=0.7080 R²=0.3410 r=0.6336 ρ=0.5954 τ=0.4194  Cliff=nan

            RAE     MAE       R2  Pearson  Spearman  Kendall  Cliff_acc
overall  0.7781  0.7080   0.3410   0.6336    0.5954   0.4194        NaN
active   5.2448  1.0998 -16.5114  -0.0040    0.0063   0.0035        NaN


In [7]:
# Final GP on all training data (full inducing set)
gp_final = GaussianProcessRegressor(kernel=C(1.0)*TanimotoKernel(),
                                      alpha=0.5, normalize_y=True)
gp_final.fit(fps_tr[inducing_idx].astype(np.float64), y_tr[inducing_idx])
te_preds_gp, te_std = gp_final.predict(fps_te.astype(np.float64), return_std=True)
te_preds = np.clip(te_preds_gp, y_tr.min()-0.5, y_tr.max()+0.5)
print(f"Test uncertainty (std): mean={te_std.mean():.3f}  max={te_std.max():.3f}")
np.save(DATA_PROCESSED/"te_gp_uncertainty.npy", te_std)
np.save(DATA_PROCESSED/"oof_gp_tanimoto.npy", oof)
np.save(DATA_PROCESSED/"te_oof_gp_tanimoto.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"77_gp_tanimoto.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


Test uncertainty (std): mean=0.753  max=0.790
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\77_gp_tanimoto.csv
Test: min=3.26 med=4.53 max=5.43
